In [1]:
import yfinance as yf
import pandas as pd

tickers = ['SPY', 'QQQ', '^VIX', 'TLT', 'GLD']

data = yf.download(tickers, start='2015-01-01', end='2024-12-31')['Close']

# Rename ^VIX column to match what the main code expects
data.columns = [c.replace('^', '^') for c in data.columns]

data.to_csv('kroniq_5asset_data.csv')
print(f"Saved: kroniq_5asset_data.csv")
print(f"Shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
print(f"Date range: {data.index[0].date()} → {data.index[-1].date()}")

[*********************100%***********************]  5 of 5 completed


Saved: kroniq_5asset_data.csv
Shape: (2515, 5)
Columns: ['GLD', 'QQQ', 'SPY', 'TLT', '^VIX']
Date range: 2015-01-02 → 2024-12-30


In [2]:
import pandas as pd
import numpy as np
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# KRONIQ — Week 4 Feature Experiment: Credit Spreads (9-feature model)
# ------------------------------------------------------------
# Adds 2 new features to v3 baseline:
#   credit_spread — ICE BofA US High Yield OAS daily level
#   cs_chg        — daily change in credit spread
# Total features: 9 (up from 7 in v3)
# v3 remains the locked baseline (Sharpe 0.923, 23-day lead time)
# This notebook is exploratory — directional and diagnostic only
# Final walk-forward validation happens in Week 5
# ============================================================

# --- Load 5-asset price data ---
spy = pd.read_csv('kroniq_5asset_data.csv',
                  index_col=0, parse_dates=True)

# --- Load credit spreads ---
cs = pd.read_csv('kroniq_credit_spreads.csv',
                 index_col=0, parse_dates=True)
cs.columns = ['credit_spread']
cs = cs.loc['2015-01-01':'2024-12-31']

# --- Pre-dropna alignment diagnostics on full market calendar ---
credit_aligned_full = cs.reindex(spy.index)
credit_missing_full = credit_aligned_full['credit_spread'].isna().sum()
print("=== Credit spread alignment diagnostics (pre-dropna) ===")
print(f"Total market rows:                        {len(spy)}")
print(f"Total credit spread rows:                 {len(cs)}")
print(f"Credit spread missing on market calendar: {credit_missing_full}")
print(f"Overlapping rows on market calendar:      "
      f"{credit_aligned_full['credit_spread'].notna().sum()}")

# --- Recreate exact same 7 features as v3 ---
spy['spy_ret']      = spy['SPY'].pct_change(fill_method=None)
spy['spy_vol']      = spy['spy_ret'].rolling(20).std() * np.sqrt(252)
spy['vix']          = spy['^VIX']
spy['vix_chg']      = spy['^VIX'].pct_change(fill_method=None)
spy['tlt_ret']      = spy['TLT'].pct_change(fill_method=None)
spy['gld_ret']      = spy['GLD'].pct_change(fill_method=None)
spy['spy_tlt_corr'] = spy['spy_ret'].rolling(20).corr(spy['tlt_ret'])

# --- Add 2 new credit spread features ---
spy['credit_spread'] = cs['credit_spread']
spy['cs_chg']        = spy['credit_spread'].diff()

# --- Feature matrix ---
feature_cols = ['spy_ret', 'spy_vol', 'vix', 'vix_chg',
                'tlt_ret', 'gld_ret', 'spy_tlt_corr',
                'credit_spread', 'cs_chg']

# --- Missing value diagnostics before dropna ---
print("\n=== Missing values by feature before dropna ===")
print(spy[feature_cols].isna().sum())
rows_before = len(spy[feature_cols])

features = spy[feature_cols].dropna()
features = features.loc['2015-01-01':'2024-12-31']

print(f"\nRows before dropna: {rows_before}")
print(f"Rows after dropna:  {len(features)}")

# --- Train/test split — no overlap ---
split_date     = '2020-12-31'
train          = features[features.index <= split_date]
test           = features[features.index > split_date]

print(f"\nTrain: {train.index[0].date()} → "
      f"{train.index[-1].date()} ({len(train)} days)")
print(f"Test:  {test.index[0].date()} → "
      f"{test.index[-1].date()} ({len(test)} days)")

scaler  = StandardScaler()
X_train = scaler.fit_transform(train)
X_all   = scaler.transform(features)

# --- BIC selection K=2..6 ---
print("\n=== BIC Selection ===")
bic_scores = {}
models     = {}
d          = X_train.shape[1]

for k in range(2, 7):
    model = GaussianHMM(n_components=k,
                        covariance_type='full',
                        n_iter=200,
                        random_state=42)
    model.fit(X_train)

    log_ll   = model.score(X_train)
    n_params = ((k - 1) +
                k * (k - 1) +
                k * d +
                k * (d * (d + 1) // 2))
    bic = -2 * log_ll + n_params * np.log(len(X_train))

    bic_scores[k] = bic
    models[k]     = model
    print(f"K={k}: BIC={bic:.1f}  LogLL={log_ll:.1f}  Params={n_params}")

optimal_k = min(bic_scores, key=bic_scores.get)
print(f"\nOptimal K (v4 9-feature model): {optimal_k}")
print(f"v3 baseline optimal K was:      4")

# --- Decode states ---
best_model     = models[optimal_k]
raw_states     = best_model.predict(X_all)
states         = pd.Series(raw_states, index=features.index, name='state')
train_states   = states.loc[train.index]
train_features = features.loc[train.index]

# --- State profiles ---
print("\n=== State profiles (training data) ===")
print(f"{'State':<8} {'Ann.Ret':>8} {'VIX':>7} "
      f"{'Cr.Spread':>11} {'Days':>6}")
print("-" * 48)

state_profiles = {}
for s in range(optimal_k):
    mask    = train_states == s
    ann_ret = train_features.loc[mask, 'spy_ret'].mean() * 252
    avg_vix = train_features.loc[mask, 'vix'].mean()
    avg_cs  = train_features.loc[mask, 'credit_spread'].mean()
    days    = mask.sum()
    state_profiles[s] = {
        'mean_ret': ann_ret,
        'mean_vix': avg_vix,
        'mean_cs':  avg_cs,
        'days':     days
    }
    print(f"{s:<8} {ann_ret:>7.1%} {avg_vix:>7.1f} "
          f"{avg_cs:>10.2f}% {days:>6}")

# --- Auditable labeling diagnostics ---
print("\n=== State labeling diagnostics ===")

highest_vix = max(state_profiles, key=lambda s: state_profiles[s]['mean_vix'])
highest_cs  = max(state_profiles, key=lambda s: state_profiles[s]['mean_cs'])
lowest_vix  = min(state_profiles, key=lambda s: state_profiles[s]['mean_vix'])
highest_ret = max(state_profiles, key=lambda s: state_profiles[s]['mean_ret'])

print(f"\n{'State':<8} {'Ann.Ret':>8} {'VIX':>7} {'Cr.Sprd':>9} "
      f"{'HighVIX':>9} {'HighCS':>8} {'LowVIX':>8} {'HighRet':>9}")
print("-" * 72)

for s, p in state_profiles.items():
    print(f"{s:<8} {p['mean_ret']:>7.1%} {p['mean_vix']:>7.1f} "
          f"{p['mean_cs']:>8.2f}% "
          f"{'YES' if s == highest_vix else 'no':>9} "
          f"{'YES' if s == highest_cs  else 'no':>8} "
          f"{'YES' if s == lowest_vix  else 'no':>8} "
          f"{'YES' if s == highest_ret else 'no':>9}")

# ============================================================
# FIXED LABELING BLOCK — handles K=5 with two stress states
# Crisis  = acute panic (highest VIX — March 2020 concentrated)
# Macro   = chronic stress (lowest return — 2022 rate shock)
# Low-Vol = lowest VIX (calm, compressed volatility)
# Bull    = highest return among remaining
# Neutral = leftover
# ============================================================

labels   = {}
assigned = set()

# Step 1 — Crisis: highest VIX (acute panic regime)
acute_crisis = max(state_profiles,
                   key=lambda s: state_profiles[s]['mean_vix'])
labels[acute_crisis] = 'Crisis'
assigned.add(acute_crisis)
print(f"\nCrisis  → State {acute_crisis} "
      f"(VIX={state_profiles[acute_crisis]['mean_vix']:.1f}, "
      f"CS={state_profiles[acute_crisis]['mean_cs']:.2f}%, "
      f"ret={state_profiles[acute_crisis]['mean_ret']:.1%}, "
      f"days={state_profiles[acute_crisis]['days']})")

# Step 2 — Macro: lowest return among remaining (chronic stress)
remaining = {s: p for s, p in state_profiles.items()
             if s not in assigned}
macro = min(remaining.items(), key=lambda x: x[1]['mean_ret'])
labels[macro[0]] = 'Macro'
assigned.add(macro[0])
print(f"Macro   → State {macro[0]} "
      f"(VIX={macro[1]['mean_vix']:.1f}, "
      f"CS={macro[1]['mean_cs']:.2f}%, "
      f"ret={macro[1]['mean_ret']:.1%}, "
      f"days={macro[1]['days']})")

# Step 3 — Low-Vol: lowest VIX among remaining
remaining = {s: p for s, p in state_profiles.items()
             if s not in assigned}
low_vol = min(remaining.items(), key=lambda x: x[1]['mean_vix'])
labels[low_vol[0]] = 'Low-Vol'
assigned.add(low_vol[0])
print(f"Low-Vol → State {low_vol[0]} "
      f"(VIX={low_vol[1]['mean_vix']:.1f}, "
      f"CS={low_vol[1]['mean_cs']:.2f}%, "
      f"ret={low_vol[1]['mean_ret']:.1%}, "
      f"days={low_vol[1]['days']})")

# Step 4 — Bull: highest return among remaining
remaining = {s: p for s, p in state_profiles.items()
             if s not in assigned}
bull = max(remaining.items(), key=lambda x: x[1]['mean_ret'])
labels[bull[0]] = 'Bull'
assigned.add(bull[0])
print(f"Bull    → State {bull[0]} "
      f"(VIX={bull[1]['mean_vix']:.1f}, "
      f"CS={bull[1]['mean_cs']:.2f}%, "
      f"ret={bull[1]['mean_ret']:.1%}, "
      f"days={bull[1]['days']})")

# Step 5 — Neutral: leftover states
remaining = {s: p for s, p in state_profiles.items()
             if s not in assigned}
for i, (s, p) in enumerate(remaining.items()):
    name = 'Neutral' if i == 0 else f'State{s}'
    labels[s] = name
    assigned.add(s)
    print(f"{name:<10} → State {s} "
          f"(VIX={p['mean_vix']:.1f}, "
          f"CS={p['mean_cs']:.2f}%, "
          f"ret={p['mean_ret']:.1%}, "
          f"days={p['days']})")

print(f"\nFinal labels: {labels}")

# --- Routing — Crisis and Macro both go to cash ---
def route(state):
    name = labels.get(state, 'Neutral')
    if name in ['Bull', 'Low-Vol']:
        return 1.0
    elif name == 'Neutral':
        return 0.7
    elif name == 'Macro':
        return 0.3
    elif name == 'Crisis':
        return 0.0
    else:
        return 0.5

oos_states       = states.loc[test.index]
oos_returns      = features.loc[test.index, 'spy_ret']
weights          = oos_states.map(route)
strategy_returns = weights.shift(1) * oos_returns

def sharpe(r):
    r = r.dropna()
    return (r.mean() / r.std()) * np.sqrt(252)

def max_dd(r):
    r   = r.dropna()
    cum = (1 + r).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min()

def cagr(r):
    r = r.dropna()
    n = len(r) / 252
    return (1 + r).prod() ** (1/n) - 1

print("\n=== OOS Results — fixed labels ===")
print(f"v4 Strategy Sharpe:     {sharpe(strategy_returns):.3f}")
print(f"v4 Buy-Hold Sharpe:     {sharpe(oos_returns):.3f}")
print(f"v4 Strategy Max DD:     {max_dd(strategy_returns):.2%}")
print(f"v4 Strategy CAGR:       {cagr(strategy_returns):.2%}")
print(f"\n--- Baseline reference (v3 locked) ---")
print(f"v3 Walk-Forward Sharpe: 0.923")
print(f"v3 Max DD:             -24.50%")

# --- Lead time analysis ---
spy_peak   = pd.Timestamp('2020-02-19')
window     = states.loc['2019-10-01':'2020-04-30']
stress_ids = [s for s, name in labels.items()
              if name in ['Crisis', 'Macro']]

print("\n=== Lead time analysis (exploratory diagnostic) ===")
if stress_ids:
    stress_window = window[window.isin(stress_ids)]
    if len(stress_window) > 0:
        first_stress = stress_window.index[0]
        lead_days    = (spy_peak - first_stress).days
        first_label  = labels[stress_window.iloc[0]]
        print(f"First stress signal (v4): {first_stress.date()} [{first_label}]")
        print(f"SPY peak:                 {spy_peak.date()}")
        print(f"Lead time (v4):           {lead_days} calendar days")
        print(f"Lead time (v3):           23 calendar days")
        if lead_days > 23:
            print(f"Credit spreads extended lead time by "
                  f"{lead_days - 23} days")
        elif lead_days == 23:
            print("Lead time unchanged — existing features "
                  "already captured signal")
        else:
            print(f"Lead time shorter by {23 - lead_days} days — investigate")
    else:
        print("No stress state found in 2019-2020 window — "
              "check state labels above")
else:
    print("No Crisis or Macro state found — check labeling above")

# --- 3 sanity checks ---
checks = [
    ('2017 calm',       '2017-01-01', '2017-12-31', 'Low-Vol', 0.60),
    ('2020 COVID peak', '2020-02-01', '2020-05-31', 'Crisis',  0.50),
    ('2022 rate shock', '2022-01-01', '2022-12-31', 'Macro',   0.80),
]

print("\n=== Sanity checks — v4 fixed labels ===")
all_passed = True
for label, start, end, expected, threshold in checks:
    s      = states.loc[start:end]
    vc     = s.map(labels).value_counts(normalize=True)
    pct    = vc.get(expected, 0)
    passed = pct >= threshold
    if not passed:
        all_passed = False
    status = "PASS" if passed else "FAIL"
    print(f"\n{label}:")
    print(f"  Expected {expected} >= {threshold:.0%} → "
          f"got {pct:.1%} [{status}]")
    print(f"  Full breakdown: {dict(vc.round(2))}")

print("\n" + ("All sanity checks passed!" if all_passed
              else "Some checks failed — investigate before Week 5."))

# --- Save outputs ---
regime_labeled = states.map(labels)
regime_labeled.to_csv('kroniq_regime_history_v4.csv')
print("\nSaved: kroniq_regime_history_v4.csv")

=== Credit spread alignment diagnostics (pre-dropna) ===
Total market rows:                        2515
Total credit spread rows:                 2612
Credit spread missing on market calendar: 1
Overlapping rows on market calendar:      2514

=== Missing values by feature before dropna ===
spy_ret           1
spy_vol          20
vix               0
vix_chg           1
tlt_ret           1
gld_ret           1
spy_tlt_corr     20
credit_spread     1
cs_chg            3
dtype: int64

Rows before dropna: 2515
Rows after dropna:  2495

Train: 2015-02-02 → 2020-12-31 (1491 days)
Test:  2021-01-04 → 2024-12-30 (1004 days)

=== BIC Selection ===


  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


K=2: BIC=26890.7  LogLL=-13039.8  Params=111
K=3: BIC=24830.2  LogLL=-11794.0  Params=170
K=4: BIC=24037.7  LogLL=-11174.9  Params=231
K=5: BIC=22867.2  LogLL=-10359.4  Params=294
K=6: BIC=23765.8  LogLL=-10571.3  Params=359

Optimal K (v4 9-feature model): 5
v3 baseline optimal K was:      4

=== State profiles (training data) ===
State     Ann.Ret     VIX   Cr.Spread   Days
------------------------------------------------
0          13.1%    14.4       4.43%    275
1          48.6%    15.5       5.49%    233
2         -18.1%    23.8       5.39%    415
3          24.2%    12.1       3.71%    521
4          26.7%    48.6       7.86%     47

=== State labeling diagnostics ===

State     Ann.Ret     VIX   Cr.Sprd   HighVIX   HighCS   LowVIX   HighRet
------------------------------------------------------------------------
0          13.1%    14.4     4.43%        no       no       no        no
1          48.6%    15.5     5.49%        no       no       no       YES
2         -18.1%    23

In [14]:
# === Week 5 Day 2 — OOS Walk-Forward Validation (final) ===
# Expanding window, quarterly retraining (~63 trading days)
# Scaler refit at each retrain step — no lookahead bias
# Viterbi decoding over full history to current day
# Train: 2015-2020 | Test: 2021-2024 | K=5, seed=42

from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

RETRAIN_FREQ  = 63
n_tr          = len(train)
n_te          = len(test)
wf_labels_out = []
scaler_wf     = None
m_wf          = None
labels_wf     = {}

print("=== Walk-Forward OOS Validation (K=5, quarterly retrain) ===")

for i in range(n_te):
    abs_i = n_tr + i

    if i == 0 or i % RETRAIN_FREQ == 0:
        print(f"  Retrained at OOS step {i} (date={test.index[i].date()})")

        scaler_wf = StandardScaler()
        X_wf      = scaler_wf.fit_transform(features.iloc[:abs_i])

        m_wf = GaussianHMM(
            n_components=5,
            covariance_type='full',
            n_iter=1000,
            random_state=42
        )
        m_wf.fit(X_wf)

        s_wf        = m_wf.predict(X_wf)
        f_wf        = features.iloc[:abs_i]
        profiles_wf = {}
        for s in range(5):
            mask = (s_wf == s)
            profiles_wf[s] = {
                'mean_ret': f_wf.loc[mask, 'spy_ret'].mean() * 252,
                'mean_vix': f_wf.loc[mask, 'vix'].mean(),
            }

        assigned_wf = set()
        labels_wf   = {}

        crisis = max(profiles_wf, key=lambda s: profiles_wf[s]['mean_vix'])
        labels_wf[crisis] = 'Crisis'
        assigned_wf.add(crisis)

        rem   = {s: p for s, p in profiles_wf.items() if s not in assigned_wf}
        macro = min(rem.items(), key=lambda x: x[1]['mean_ret'])
        labels_wf[macro[0]] = 'Macro'
        assigned_wf.add(macro[0])

        rem = {s: p for s, p in profiles_wf.items() if s not in assigned_wf}
        lv  = min(rem.items(), key=lambda x: x[1]['mean_vix'])
        labels_wf[lv[0]] = 'Low-Vol'
        assigned_wf.add(lv[0])

        rem  = {s: p for s, p in profiles_wf.items() if s not in assigned_wf}
        bull = max(rem.items(), key=lambda x: x[1]['mean_ret'])
        labels_wf[bull[0]] = 'Bull'
        assigned_wf.add(bull[0])

        rem = {s: p for s, p in profiles_wf.items() if s not in assigned_wf}
        for s, p in rem.items():
            labels_wf[s] = 'Neutral'

    # Viterbi decoding over full history to current day
    X_hist_today = scaler_wf.transform(features.iloc[:abs_i + 1])
    s_today      = m_wf.predict(X_hist_today)[-1]
    wf_labels_out.append(labels_wf.get(s_today, 'Neutral'))

# Performance
wf_series  = pd.Series(wf_labels_out, index=test.index)
oos_ret    = features.loc[test.index, 'spy_ret']
wf_weights = wf_series.map(route)
wf_strat   = wf_weights.shift(1) * oos_ret

print(f"\n=== Results ===")
print(f"Walk-Forward Sharpe:  {sharpe(wf_strat):.3f}")
print(f"Walk-Forward MaxDD:   {max_dd(wf_strat):.2%}")
print(f"Walk-Forward CAGR:    {cagr(wf_strat):.2%}")
print(f"\n--- Benchmark ---")
print(f"Buy-Hold Sharpe:      {sharpe(oos_ret):.3f}")
print(f"\n--- Static OOS reference ---")
print(f"Static Sharpe:        1.107")
print(f"Static MaxDD:         -8.37%")

=== Walk-Forward OOS Validation (K=5, quarterly retrain) ===
  Retrained at OOS step 0 (date=2021-01-04)
  Retrained at OOS step 63 (date=2021-04-06)
  Retrained at OOS step 126 (date=2021-07-06)
  Retrained at OOS step 189 (date=2021-10-04)
  Retrained at OOS step 252 (date=2022-01-03)
  Retrained at OOS step 315 (date=2022-04-04)
  Retrained at OOS step 378 (date=2022-07-06)
  Retrained at OOS step 441 (date=2022-10-04)
  Retrained at OOS step 504 (date=2023-01-04)
  Retrained at OOS step 567 (date=2023-04-05)
  Retrained at OOS step 630 (date=2023-07-07)
  Retrained at OOS step 693 (date=2023-10-05)
  Retrained at OOS step 756 (date=2024-01-05)
  Retrained at OOS step 819 (date=2024-04-08)
  Retrained at OOS step 882 (date=2024-07-09)
  Retrained at OOS step 945 (date=2024-10-07)

=== Results ===
Walk-Forward Sharpe:  0.881
Walk-Forward MaxDD:   -17.46%
Walk-Forward CAGR:    9.97%

--- Benchmark ---
Buy-Hold Sharpe:      0.859

--- Static OOS reference ---
Static Sharpe:        1.10